# Movie Recommendation System Using TensorFlow

This notebook is heavily inspiration from the [TensorFlow recommender basics tutorial series](https://www.tensorflow.org/recommenders/examples/basic_retrieval).

Real-world recommender systems are often composed of two stages:
- Retrival Stage
- Ranking Stage

## Dataset

In this notebook, we are going to use the [MovieLens dataset by GroupLens](https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset). For a comprehensive overview, you can refer to the dataset page on Kaggle.

# Imports

In [3]:
!pip install tensorflow==2.15.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 3.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 46.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 71.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.2
    Uninstalling wrapt-1.17.2:
      Successfully uninstalled wrapt-1.17.2
  Attempting uninstall: keras
    Found existing installation: keras 3.8.0
    Uninstalling keras-3.8.0:
      Successfully uninstalled keras-3.8.0
  Attempting uninstall: tensorboard
    Found existing installation: tensorboard 2.18.0
    Uninstalling tensorboard-2.18.0:
      Successfully uninstalled tensorboard-2.18.0
  Attempting uninstall

In [4]:
!pip install -q tensorflow-recommenders
!pip install -q --upgrade tensorflow-datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.1/321.1 kB 7.3 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tensorflow 2.15.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 6.31.1 which is incompatible.
google-api-core 1.34.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<4.0.0dev,>=3.19.5, but you have protobuf 6.31.1 which is incompatible.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.31.1 which is incompatible.
dopamine-rl 4.1.2 requires gym

In [5]:
import os
import pprint
import tempfile

from typing import Dict, Text

import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

2025-07-18 03:35:30.282644: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-07-18 03:35:30.283631: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-07-18 03:35:30.285876: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [6]:
tf.__version__

'2.15.1'

In [68]:
import tensorflow_recommenders as tfrs

# EDA And Preprocessing

In [69]:
# Ratings data.
ratings = tfds.load("movielens/100k-ratings", split="train")
# Features of all the available movies.
movies = tfds.load("movielens/100k-movies", split="train")

Check the values

In [70]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'bucketized_user_age': 45.0,
 'movie_genres': array([7]),
 'movie_id': b'357',
 'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'timestamp': 879024327,
 'user_gender': True,
 'user_id': b'138',
 'user_occupation_label': 4,
 'user_occupation_text': b'doctor',
 'user_rating': 4.0,
 'user_zip_code': b'53211'}


In [71]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'movie_genres': array([4]),
 'movie_id': b'1681',
 'movie_title': b'You So Crazy (1994)'}


## Feature Selection
We are only intersted in the user preference

In [72]:
# ratings = ratings.map(lambda x: {
#     "movie_title": x["movie_title"],
#     "user_id": x["user_id"],
# })
ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
    "raw_user_age": x["raw_user_age"],
    "user_rating": x["user_rating"]
})
movies = movies.map(lambda x: x["movie_title"])

Print data after feature selection

In [73]:
for x in ratings.take(1).as_numpy_iterator():
  pprint.pprint(x)

{'movie_title': b"One Flew Over the Cuckoo's Nest (1975)",
 'raw_user_age': 46.0,
 'user_id': b'138',
 'user_rating': 4.0}


In [74]:
for x in movies.take(1).as_numpy_iterator():
  pprint.pprint(x)

b'You So Crazy (1994)'


## Train & Test Split

In [75]:
tf.random.set_seed(42)
train = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

## Number of unique users and movies

In [76]:
movie_titles = movies.batch(1_000)
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

unique_movie_titles[:10]

array([b"'Til There Was You (1997)", b'1-900 (1994)',
       b'101 Dalmatians (1996)', b'12 Angry Men (1957)', b'187 (1997)',
       b'2 Days in the Valley (1996)',
       b'20,000 Leagues Under the Sea (1954)',
       b'2001: A Space Odyssey (1968)',
       b'3 Ninjas: High Noon At Mega Mountain (1998)',
       b'39 Steps, The (1935)'], dtype=object)

In [77]:
len(unique_user_ids)

943

# Two-tower Retrieval Model

Retrieval models are often composed of two sub-models:

1. A query model computing the query representation (normally a fixed-dimensionality embedding vector) using query features.
2. A candidate model computing the candidate representation (an equally-sized vector) using the candidate features

The outputs of the two models are then multiplied together to give a query-candidate affinity score, with higher scores expressing a better match between the candidate and the query.

In [78]:
embedding_dimension = 32

## Query Sub-Model

In [79]:
user_model = tf.keras.Sequential([
  tf.keras.layers.StringLookup(
      vocabulary=unique_user_ids, mask_token=None),
  # We add an additional embedding to account for unknown tokens.
  tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)
])

## Candidate Sub-Model

In [80]:
movie_model = tf.keras.Sequential([
  tf.keras.layers.StringLookup(
      vocabulary=unique_movie_titles, mask_token=None),
  tf.keras.layers.Embedding(len(unique_movie_titles) + 1, embedding_dimension)
])

## Metrics

In [81]:
# Preformance
metrics = tfrs.metrics.FactorizedTopK(
  candidates=movies.batch(128).map(movie_model)
)

In [82]:
# Loss
task = tfrs.tasks.Retrieval(
  metrics=metrics
)

## Architecture

In [83]:
class MovielensModel(tfrs.Model):
  def __init__(self, user_model, movie_model):
    super().__init__()
    self.movie_model: tf.keras.Model = movie_model
    self.user_model: tf.keras.Model = user_model
    self.task: tf.keras.layers.Layer = task

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    # We pick out the user features and pass them into the user model.
    user_embeddings = self.user_model(features["user_id"])
    # And pick out the movie features and pass them into the movie model,
    # getting embeddings back.
    positive_movie_embeddings = self.movie_model(features["movie_title"])

    # The task computes the loss and the metrics.
    return self.task(user_embeddings, positive_movie_embeddings)

In [84]:
# # Create a retrieval model.
# model = MovieLensModel(user_model, movie_model, task)
# model.compile(optimizer=tf.keras.optimizers.Adagrad(0.5))

# # Train for 3 epochs.
# model.fit(ratings.batch(4096), epochs=3)

# # Use brute-force search to set up retrieval using the trained representations.
# index = tfrs.layers.factorized_top_k.BruteForce(model.user_model)
# index.index_from_dataset(
#     movies.batch(100).map(lambda title: (title, model.movie_model(title))))

# # Get some recommendations.
# _, titles = index(np.array(["42"]))
# print(f"Top 3 recommendations for user 42: {titles[0, :3]}")

In [85]:
retrival_model = MovielensModel(user_model, movie_model)
retrival_model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

In [86]:
cached_train = train.shuffle(100_000).batch(8192).cache()

In [87]:
retrival_model.fit(cached_train, epochs=3)

Epoch 1/3
13/13 [==============================] - 27s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0014 - factorized_top_k/top_5_categorical_accuracy: 0.0111 - factorized_top_k/top_10_categorical_accuracy: 0.0231 - factorized_top_k/top_50_categorical_accuracy: 0.1132 - factorized_top_k/top_100_categorical_accuracy: 0.1997 - loss: 64357.6265 - regularization_loss: 0.0000e+00 - total_loss: 64357.6265
Epoch 2/3
13/13 [==============================] - 26s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0029 - factorized_top_k/top_5_categorical_accuracy: 0.0187 - factorized_top_k/top_10_categorical_accuracy: 0.0378 - factorized_top_k/top_50_categorical_accuracy: 0.1660 - factorized_top_k/top_100_categorical_accuracy: 0.2882 - loss: 62128.3001 - regularization_loss: 0.0000e+00 - total_loss: 62128.3001
Epoch 3/3
13/13 [==============================] - 23s 2s/step - factorized_top_k/top_1_categorical_accuracy: 0.0030 - factorized_top_k/top_5_categorical_accuracy: 0.0216

In [88]:
retrival_model.evaluate(cached_train, return_dict=True)

13/13 [==============================] - 19s 1s/step - factorized_top_k/top_1_categorical_accuracy: 0.0048 - factorized_top_k/top_5_categorical_accuracy: 0.0290 - factorized_top_k/top_10_categorical_accuracy: 0.0556 - factorized_top_k/top_50_categorical_accuracy: 0.2093 - factorized_top_k/top_100_categorical_accuracy: 0.3397 - loss: 60553.4407 - regularization_loss: 0.0000e+00 - total_loss: 60553.4407


{'factorized_top_k/top_1_categorical_accuracy': 0.004769999999552965,
 'factorized_top_k/top_5_categorical_accuracy': 0.029020000249147415,
 'factorized_top_k/top_10_categorical_accuracy': 0.05559999868273735,
 'factorized_top_k/top_50_categorical_accuracy': 0.20934000611305237,
 'factorized_top_k/top_100_categorical_accuracy': 0.339709997177124,
 'loss': 11402.9990234375,
 'regularization_loss': 0,
 'total_loss': 11402.9990234375}

## Inference

In [89]:
# Create a model that takes in raw query features, and
index = tfrs.layers.factorized_top_k.BruteForce(retrival_model.user_model)
# recommends movies out of the entire movies dataset.
index.index_from_dataset(
  tf.data.Dataset.zip((movies.batch(100), movies.batch(100).map(retrival_model.movie_model)))
)

# Get recommendations, user 42
_, titles = index(tf.constant(["42"]))
print(f"Recommendations for user 42: {titles[0, :3]}")

Recommendations for user 42: [b'Angels in the Outfield (1994)' b'Rudy (1993)'
 b'Father of the Bride Part II (1995)']


In [90]:
titles

<tf.Tensor: shape=(1, 10), dtype=string, numpy=
array([[b'Angels in the Outfield (1994)', b'Rudy (1993)',
        b'Father of the Bride Part II (1995)', b'Just Cause (1995)',
        b'Indian in the Cupboard, The (1995)', b'Lion King, The (1994)',
        b'Rent-a-Kid (1995)', b'Fried Green Tomatoes (1991)',
        b'Ghost (1990)', b'Bridges of Madison County, The (1995)']],
      dtype=object)>

In [91]:
tf.strings.length(titles)

<tf.Tensor: shape=(1, 10), dtype=int32, numpy=array([[29, 11, 34, 17, 34, 21, 17, 27, 12, 37]], dtype=int32)>

In [92]:
# Get recommendations, user 2
_, titles = index(tf.constant(["2"]))
print(f"Recommendations for user 2: {titles[0, :3]}")

Recommendations for user 2: [b'A Chef in Love (1996)' b'Kolya (1996)' b'Restoration (1995)']


In [93]:
titles

<tf.Tensor: shape=(1, 10), dtype=string, numpy=
array([[b'A Chef in Love (1996)', b'Kolya (1996)', b'Restoration (1995)',
        b'Beautiful Thing (1996)', b"Antonia's Line (1995)",
        b'Brassed Off (1996)', b'Wild Reeds (1994)',
        b'Secrets & Lies (1996)', b'Sense and Sensibility (1995)',
        b"Ulee's Gold (1997)"]], dtype=object)>

In [94]:
tf.strings.length(titles)

<tf.Tensor: shape=(1, 10), dtype=int32, numpy=array([[21, 12, 18, 22, 21, 18, 17, 21, 28, 18]], dtype=int32)>

In [95]:
# Get recommendations, user 2
_, titles = index(tf.constant(["200"]))
print(f"Recommendations for user 200: {titles[0, :3]}")

Recommendations for user 200: [b"Kid in King Arthur's Court, A (1995)" b'Goofy Movie, A (1995)'
 b'First Kid (1996)']


In [96]:
titles

<tf.Tensor: shape=(1, 10), dtype=string, numpy=
array([[b"Kid in King Arthur's Court, A (1995)",
        b'Goofy Movie, A (1995)', b'First Kid (1996)',
        b'Homeward Bound II: Lost in San Francisco (1996)',
        b'Jungle Book, The (1994)',
        b'Aladdin and the King of Thieves (1996)', b'Quest, The (1996)',
        b"Pete's Dragon (1977)", b'Casper (1995)', b'Pocahontas (1995)']],
      dtype=object)>

# Ranking Model

## Data Preparation

In [97]:
ratings = tfds.load("movielens/100k-ratings", split="train")

ratings = ratings.map(lambda x: {
    "movie_title": x["movie_title"],
    "user_id": x["user_id"],
    "raw_user_age": x["raw_user_age"],
    "user_rating": x["user_rating"]
})

In [98]:
user_ids = set()

for item in ratings:
    user_ids.add(item['user_id'].numpy())  # Convert tensor to Python value

print(f"Number of unique users: {len(user_ids)}")


Number of unique users: 943


In [99]:
# expected_user_ids = set(range(1, 944))  # from 1 to 943 (inclusive)
# actual_user_ids = set()

# for item in ratings:
#     user_id = item["user_id"].numpy()  # Convert tf.Tensor to int
#     actual_user_ids.add(user_id)

# # Check if the sets match
# if actual_user_ids == expected_user_ids:
#     print("✅ User IDs are sequential from 1 to 943 with no gaps.")
# else:
#     print("❌ User IDs are not sequential from 1 to 943.")
#     print(f"Missing IDs: {sorted(expected_user_ids - actual_user_ids)}")
#     print(f"Extra IDs: {sorted(actual_user_ids - expected_user_ids)}")


In [100]:
tf.random.set_seed(42)
shuffled = ratings.shuffle(100_000, seed=42, reshuffle_each_iteration=False)

train = shuffled.take(80_000)
test = shuffled.skip(80_000).take(20_000)

In [101]:
len(test)

20000

In [102]:
test

<_TakeDataset element_spec={'movie_title': TensorSpec(shape=(), dtype=tf.string, name=None), 'user_id': TensorSpec(shape=(), dtype=tf.string, name=None), 'raw_user_age': TensorSpec(shape=(), dtype=tf.float32, name=None), 'user_rating': TensorSpec(shape=(), dtype=tf.float32, name=None)}>

In [103]:
print(train)

<_TakeDataset element_spec={'movie_title': TensorSpec(shape=(), dtype=tf.string, name=None), 'user_id': TensorSpec(shape=(), dtype=tf.string, name=None), 'raw_user_age': TensorSpec(shape=(), dtype=tf.float32, name=None), 'user_rating': TensorSpec(shape=(), dtype=tf.float32, name=None)}>


In [104]:
for example in test.take(1):
    print("Keys available in example:", example.keys())


Keys available in example: dict_keys(['movie_title', 'user_id', 'raw_user_age', 'user_rating'])


In [105]:
target_user = "17"

for item in test:
    user_id = item['user_id'].numpy().decode('utf-8')
    if user_id == target_user:
        result = {
            'movie_title': item['movie_title'].numpy().decode('utf-8'),
            'user_id': user_id,
            'user_rating': float(item['user_rating'].numpy()) if 'user_rating' in item else None
        }
        print(result)


{'movie_title': 'Phenomenon (1996)', 'user_id': '17', 'user_rating': 1.0}
{'movie_title': "Devil's Own, The (1997)", 'user_id': '17', 'user_rating': 2.0}
{'movie_title': 'Fargo (1996)', 'user_id': '17', 'user_rating': 4.0}
{'movie_title': 'Twelve Monkeys (1995)', 'user_id': '17', 'user_rating': 4.0}
{'movie_title': 'Trainspotting (1996)', 'user_id': '17', 'user_rating': 4.0}
{'movie_title': 'Breaking the Waves (1996)', 'user_id': '17', 'user_rating': 2.0}


In [106]:
# import pandas as pd

# data = {
#     'movie_title': [],
#     'user_id': [],
#     'user_rating': [],
#     'raw_user_age':[]
# }

# for example in dataset:
#     data['movie_title'].append(example['movie_title'].numpy().decode('utf-8'))
#     data['user_id'].append(example['user_id'].numpy().decode('utf-8'))
#     data['user_rating'].append(float(example['user_rating'].numpy()))

# df = pd.DataFrame(data)
# df.to_csv('output.csv', index=False)

In [107]:
movie_titles = ratings.batch(1_000_000).map(lambda x: x["movie_title"])
user_ids = ratings.batch(1_000_000).map(lambda x: x["user_id"])

unique_movie_titles = np.unique(np.concatenate(list(movie_titles)))
unique_user_ids = np.unique(np.concatenate(list(user_ids)))

## Architecture

In [108]:
class RankingModel(tf.keras.Model):

  def __init__(self):
    super().__init__()
    embedding_dimension = 32

    # Compute embeddings for users.
    self.user_embeddings = tf.keras.Sequential([
      tf.keras.layers.StringLookup(
        vocabulary=unique_user_ids, mask_token=None),
      tf.keras.layers.Embedding(len(unique_user_ids) + 1, embedding_dimension)
    ])

    # Compute embeddings for movies.
    self.movie_embeddings = tf.keras.Sequential([
      tf.keras.layers.StringLookup(
        vocabulary=unique_movie_titles, mask_token=None),
      tf.keras.layers.Embedding(len(unique_movie_titles) + 1, embedding_dimension)
    ])

    # Compute predictions.
    self.ratings = tf.keras.Sequential([
      # Learn multiple dense layers.
      tf.keras.layers.Dense(256, activation="relu"),
      tf.keras.layers.Dense(64, activation="relu"),
      # Make rating predictions in the final layer.
      tf.keras.layers.Dense(1)
    ])

  def call(self, inputs):

    user_id, movie_title = inputs

    user_embedding = self.user_embeddings(user_id)
    movie_embedding = self.movie_embeddings(movie_title)

    return self.ratings(tf.concat([user_embedding, movie_embedding], axis=1))

In [109]:
# This model takes user ids and movie titles, and outputs a predicted rating
RankingModel()((["42"], ["One Flew Over the Cuckoo's Nest (1975)"]))

<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[-0.00119704]], dtype=float32)>

## Loss and metrics

In [110]:
rating_task = tfrs.tasks.Ranking(
  loss = tf.keras.losses.MeanSquaredError(),
  metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

## Full Model Implementation

In [111]:
class MovielensModel(tfrs.models.Model):

  def __init__(self):
    super().__init__()
    self.ranking_model: tf.keras.Model = RankingModel()
    self.task: tf.keras.layers.Layer = rating_task

  def call(self, features: Dict[str, tf.Tensor]) -> tf.Tensor:
    return self.ranking_model(
        (features["user_id"], features["movie_title"]))

  def compute_loss(self, features: Dict[Text, tf.Tensor], training=False) -> tf.Tensor:
    labels = features.pop("user_rating")

    rating_predictions = self(features)

    # The task computes the loss and the metrics.
    return self.task(labels=labels, predictions=rating_predictions)

In [112]:
ranking_model = MovielensModel()
ranking_model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))

In [113]:
cached_train = train.shuffle(100_000).batch(8192).cache()
cached_test = test.batch(4096).cache()

In [114]:
ranking_model.fit(cached_train, epochs=3)

Epoch 1/3
10/10 [==============================] - 4s 59ms/step - root_mean_squared_error: 2.0836 - loss: 4.0159 - regularization_loss: 0.0000e+00 - total_loss: 4.0159
Epoch 2/3
10/10 [==============================] - 0s 27ms/step - root_mean_squared_error: 1.1606 - loss: 1.3404 - regularization_loss: 0.0000e+00 - total_loss: 1.3404
Epoch 3/3
10/10 [==============================] - 0s 24ms/step - root_mean_squared_error: 1.1112 - loss: 1.2359 - regularization_loss: 0.0000e+00 - total_loss: 1.2359


In [115]:
ranking_model.evaluate(cached_test, return_dict=True)

5/5 [==============================] - 3s 27ms/step - root_mean_squared_error: 1.1007 - loss: 1.2063 - regularization_loss: 0.0000e+00 - total_loss: 1.2063


{'root_mean_squared_error': 1.1006500720977783,
 'loss': 1.1840689182281494,
 'regularization_loss': 0,
 'total_loss': 1.1840689182281494}

## Inference

In [116]:
test_ratings = {}
test_movie_titles = ["M*A*S*H (1970)", "Dances with Wolves (1990)", "Speed (1994)"]
for movie_title in test_movie_titles:
  test_ratings[movie_title] = ranking_model({
      "user_id": np.array(["42"]),
      "movie_title": np.array([movie_title])
  })

print("Ratings:")
for title, score in sorted(test_ratings.items(), key=lambda x: x[1], reverse=True):
  print(f"{title}: {score}")

Ratings:
Dances with Wolves (1990): [[3.5566978]]
Speed (1994): [[3.523561]]
M*A*S*H (1970): [[3.5137823]]


# Full Recommder System

Combine the retrival model with the ranking model to provide accurate recommendations.

In [117]:
class Recommender():
    def __init__(self, retrival_model, ranking_model):
        index = tfrs.layers.factorized_top_k.BruteForce(retrival_model.user_model)
        index.index_from_dataset(
          tf.data.Dataset.zip((movies.batch(100), movies.batch(100).map(retrival_model.movie_model)))
        )
        self.retrival_model = index
        self.ranking_model = ranking_model
    
    def predict(self, user_id):
        _, titles = self.retrival_model(tf.constant([user_id]))
        ratings = []
        for index, title in enumerate(titles[0]):
            ratings.append(self.ranking_model({
                "user_id": np.array([user_id]),
                "movie_title": np.array([title.numpy().decode('utf-8')])
            }))
        return sorted(zip(titles.numpy().tolist()[0], tf.squeeze(tf.concat(ratings, axis=0)).numpy().tolist()), key=lambda x: x[1], reverse=True)

In [118]:
recommender_model = Recommender(retrival_model, ranking_model)

In [122]:
recommender_model.predict("7")

[(b'That Old Feeling (1997)', 3.5913584232330322),
 (b'For the Moment (1994)', 3.5422439575195312),
 (b"My Best Friend's Wedding (1997)", 3.4669885635375977),
 (b'Selena (1997)', 3.4596736431121826),
 (b'To Gillian on Her 37th Birthday (1996)', 3.4577670097351074),
 (b'Associate, The (1996)', 3.454542398452759),
 (b'In Love and War (1996)', 3.4442834854125977),
 (b'Thin Line Between Love and Hate, A (1996)', 3.436724901199341),
 (b'Juror, The (1996)', 3.436532974243164),
 (b'One Fine Day (1996)', 3.4233908653259277)]